# **Stage 1 — Macro Variable Forecasting**
## SARIMA Pipeline | US Quarterly
___
**Purpose:** Forecast the 12 raw macroeconomic series from the EDA analytical
dataset, reconstruct derived variables from those forecasts, and assemble the
complete historical + forecast regressor dataset used as input for Stage 2
PD modelling.

**Data source:** Reads directly from the project GitHub repository — no local
paths required. Outputs are saved to the local `Data Collection/` folder and
should be committed to the repository after running.

| Step | Description | Output |
|------|-------------|--------|
| 0 | Configuration & data loading | — |
| 1 | SARIMA pipeline functions | — |
| 2 | Run SARIMA on all 12 raw series | — |
| 3 | Derive transformed variables from forecasts | — |
| 4 | Assemble complete regressor dataset | `SARIMA_regressors_US_Q.csv` |
| 5 | Forecast summary & validation | — |

**Repository:** `hogandan85/ST-498` · branch `main`

**References:**
Bank & Eder (2021). *A Review on the Probability of Default for IFRS 9.* SSRN 3981339.
Djeundje & Crook (2019). Dynamic survival models with varying coefficients for credit risks.
*European Journal of Operational Research*, 275(1), 319–333.

## **0: Configuration & Data Loading**
___
All user-facing settings live here. To change the variable selection, forecast
horizon, lag structure, or output filenames — edit this cell only. The pipeline
functions in Section 1 are never modified directly.

**Data is loaded directly from the GitHub raw URL** — the repo must be public
or a personal access token must be set in `GITHUB_TOKEN`. No local path
configuration is needed to run this notebook on any machine.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.special import inv_boxcox
from sklearn.preprocessing import PowerTransformer
from pmdarima import auto_arima
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── GitHub data source ────────────────────────────────────────────────────────
GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/hogandan85/ST-498/main/Data%20Collection'
GITHUB_TOKEN    = None

# ── Local output directory ────────────────────────────────────────────────────
LOCAL_OUTPUT = Path.cwd().parent / 'Data Collection'

# ── Forecast horizon ──────────────────────────────────────────────────────────
# 20 quarters = 5 years, consistent with IFRS 9 lifetime ECL horizon
N_PERIODS = 20
M         = 4

# ── Raw series to forecast ────────────────────────────────────────────────────
# Level/index series only. Derived variables are reconstructed from these
# forecasts in Section 3 — do not add derived cols here.
RAW_TO_FORECAST = [
    'us_real_gdp',
    'us_unemployment',
    'us_cpi',
    'us_consumer_confidence',
    'us_bond_yield_10y',
    'us_credit',
    'us_sp500_close',
    'us_vix',
    'us_house_price_idx',
    'us_industrial_production',
    'us_oil_price',
    'us_reer',
]

# ── Derived variable construction ─────────────────────────────────────────────
# Each entry: output_col -> (source_col, method, periods)
# method: 'pct_change_yoy' | 'pct_change_qoq' | 'diff' | 'log_diff'
DERIVED_VARS = {
    'us_gdp_yoy_growth':    ('us_real_gdp',              'pct_change_yoy', 4),
    'us_house_price_yoy':   ('us_house_price_idx',       'pct_change_yoy', 4),
    'us_indprod_yoy':       ('us_industrial_production',  'pct_change_yoy', 4),
    'us_oil_yoy':           ('us_oil_price',             'pct_change_yoy', 4),
    'us_reer_diff':         ('us_reer',                  'diff',           1),
    'us_credit_qoq_growth': ('us_credit',                'pct_change_qoq', 1),
    'us_sp500_log_ret':     ('us_sp500_close',           'log_diff',       1),
    'us_vix_log_ret':       ('us_vix',                   'log_diff',       1),
    'us_bond_yield_10y_d1': ('us_bond_yield_10y',        'diff',           1),
}

# ── Lag specification ─────────────────────────────────────────────────────────
# CCF-optimal lags from EDA Section 5
# Fast-transmitting variables: max 4Q window
# Slow-transmitting variables (house prices, credit, CPI): max 6Q window
LAG_SPEC = {
    'us_house_price_yoy':     3,   
    'us_consumer_confidence': 2,   
    'us_unemployment':        1,  
    'us_credit_qoq_growth':   6,   
    'us_gdp_yoy_growth':      0,   
    'us_cpi':                 6,   
    'us_bond_yield_10y_d1':   2,
    'us_sp500_log_ret':       4,
    'us_reer_diff':           1,
    'us_indprod_yoy':         3,
    'us_oil_yoy':             2,
    'us_vix_log_ret':         0,
}

# ── Output filenames ──────────────────────────────────────────────────────────
# When adding SARIMAX or ML models, change the prefix only.
MODEL_PREFIX       = 'SARIMA'
OUT_REGRESSORS = f'{MODEL_PREFIX}_regressors_US_Q.csv'

# ── Colour palette (Plotly) ───────────────────────────────────────────────────
NAVY, BLUE, RED, AMBER, GREEN, GREY = (
    '#1F3864', '#2E75B6', '#C0392B', '#E67E22', '#27AE60', '#7F8C8D')
TEMPLATE = 'plotly_white'
RECESSIONS_US = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]

# ── Load EDA analytical dataset from GitHub ───────────────────────────────────
def load_github_csv(filename: str, token: str = None) -> pd.DataFrame:
    url = f'{GITHUB_RAW_BASE}/{filename}'
    if token:
        import requests, io
        headers = {'Authorization': f'token {token}'}
        r = requests.get(url, headers=headers)
        r.raise_for_status()
        return pd.read_csv(io.StringIO(r.text), index_col=0, parse_dates=True)
    return pd.read_csv(url, index_col=0, parse_dates=True)

df_hist = load_github_csv('EDA_analytical_US_Q.csv', token=GITHUB_TOKEN)

print(f'Loaded EDA_analytical_US_Q.csv from GitHub')
print(f'  Shape  : {df_hist.shape}')
print(f'  From   : {df_hist.index.min().date()}  to  {df_hist.index.max().date()}')
print(f'  Target : us_delinquency_rate — '
      f'{df_hist["us_delinquency_rate"].notna().sum()} obs | '
      f'min {df_hist["us_delinquency_rate"].min():.2f}% | '
      f'max {df_hist["us_delinquency_rate"].max():.2f}% | '
      f'mean {df_hist["us_delinquency_rate"].mean():.2f}%')
print(f'\nRaw series to forecast ({len(RAW_TO_FORECAST)}):')
for col in RAW_TO_FORECAST:
    status = 'OK' if col in df_hist.columns else 'MISSING'
    nans   = df_hist[col].isna().sum() if col in df_hist.columns else '—'
    print(f'  [{status}]  {col:<35s}  NaN={nans}')

Loaded EDA_analytical_US_Q.csv from GitHub
  Shape  : (145, 40)
  From   : 1990-03-31  to  2026-03-31
  Target : us_delinquency_rate — 140 obs | min 1.53% | max 6.77% | mean 3.70%

Raw series to forecast (12):
  [OK]  us_real_gdp                          NaN=1
  [OK]  us_unemployment                      NaN=0
  [OK]  us_cpi                               NaN=0
  [OK]  us_consumer_confidence               NaN=0
  [OK]  us_bond_yield_10y                    NaN=0
  [OK]  us_credit                            NaN=2
  [OK]  us_sp500_close                       NaN=0
  [OK]  us_vix                               NaN=0
  [OK]  us_house_price_idx                   NaN=1
  [OK]  us_industrial_production             NaN=0
  [OK]  us_oil_price                         NaN=0
  [OK]  us_reer                              NaN=16


## **1: SARIMA Pipeline Functions**
___
Core pipeline functions. These are never called directly — they are invoked
by `run_sarima_pipeline()` in Section 2.

**Design note:** The transformation and fitting logic is kept separate from
the dataset assembly logic so that other forecasting approaches (SARIMAX, ML)
can plug into the same assembly pipeline in Section 3–4 by returning a
`results` dict in the same format.

In [2]:
# ── Transformation helpers ────────────────────────────────────────────────────

def apply_transformations(series: pd.Series) -> dict:
    """Returns dict of {name: (transformed_series, inverse_fn)}."""
    transformations = {}
    has_negatives = (series <= 0).any()

    if not has_negatives:
        log_series = np.log(series)
        transformations['log'] = (log_series, lambda x: np.exp(x))

    if not has_negatives:
        bc_transformed, lam = stats.boxcox(series)
        bc_series = pd.Series(bc_transformed, index=series.index)
        transformations['boxcox'] = (bc_series, lambda x, l=lam: inv_boxcox(x, l))

    pt = PowerTransformer(method='yeo-johnson')
    yj_vals = pt.fit_transform((series + 0.0001).values.reshape(-1, 1)).flatten()
    yj_series = pd.Series(yj_vals, index=series.index)
    transformations['yeo-johnson'] = (
        yj_series,
        lambda x: pt.inverse_transform(np.array(x).reshape(-1, 1)).flatten() - 0.0001
    )
    return transformations


def extract_diagnostics(model) -> dict:
    """Extracts Ljung-Box, Jarque-Bera, heteroskedasticity stats from fitted model."""
    try:
        sarimax_model = model.arima_res_
        diag = {
            'ljung_box_q':    sarimax_model.test_serial_correlation('ljungbox', lags=1)[0][1][0],
            'jarque_bera_jb': sarimax_model.test_normality('jarquebera')[0][1],
            'heterosked_h':   sarimax_model.test_heteroskedasticity('breakvar')[0][1],
        }
        resid = sarimax_model.resid
        from scipy.stats import skew, kurtosis
        return {
            'prob_q':   round(diag['ljung_box_q'], 4),
            'prob_jb':  round(diag['jarque_bera_jb'], 4),
            'prob_h':   round(diag['heterosked_h'], 4),
            'skew':     round(float(skew(resid)), 4),
            'kurtosis': round(float(kurtosis(resid, fisher=False)), 4),
        }
    except Exception:
        return {'prob_q': None, 'prob_jb': None, 'prob_h': None,
                'skew': None, 'kurtosis': None}


# ── Model fitting ─────────────────────────────────────────────────────────────

def fit_all_transformations(series: pd.Series, m: int = 4) -> pd.DataFrame:
    """
    Fits auto_arima for each eligible transformation.
    Returns a comparison DataFrame sorted by AIC (best first).
    """
    transformations = apply_transformations(series)
    results = []

    for name, (transformed, inverse_fn) in transformations.items():
        try:
            model = auto_arima(
                y=transformed,
                start_p=0, start_q=0, max_p=3, max_q=3,
                start_P=0, start_Q=0, max_P=2, max_Q=2,
                m=m, d=None, D=None, seasonal=True,
                information_criterion='aic',
                error_action='ignore', suppress_warnings=True, stepwise=True
            )
            residuals   = pd.Series(model.resid())
            diagnostics = extract_diagnostics(model)
            results.append({
                'transformation': name,
                'aic':            model.aic(),
                'bic':            model.bic(),
                'order':          model.order,
                'seasonal_order': model.seasonal_order,
                'residual_std':   residuals.std(),
                'abs_skew':       abs(residuals.skew()),
                'abs_kurt':       abs(residuals.kurtosis() - 3),
                'prob_q':         diagnostics['prob_q'],
                'prob_jb':        diagnostics['prob_jb'],
                'prob_h':         diagnostics['prob_h'],
                'skew':           diagnostics['skew'],
                'kurtosis':       diagnostics['kurtosis'],
                '_model':         model,
                '_inverse_fn':    inverse_fn,
            })
        except Exception as e:
            print(f'    ⚠ Failed [{name}]: {e}')

    return pd.DataFrame(results).sort_values('aic').reset_index(drop=True)


def forecast_best_model(comparison_df: pd.DataFrame,
                         n_periods: int = 20) -> dict:
    """Takes comparison df, selects best model by AIC, returns forecast."""
    best       = comparison_df.iloc[0]
    model      = best['_model']
    inverse_fn = best['_inverse_fn']
    return {
        'transformation': best['transformation'],
        'order':          best['order'],
        'seasonal_order': best['seasonal_order'],
        'aic':            best['aic'],
        'forecast':       inverse_fn(model.predict(n_periods=n_periods)),
        'model':          model,
        'inverse_fn':     inverse_fn,
    }


# ── Main pipeline ─────────────────────────────────────────────────────────────

def run_sarima_pipeline(df: pd.DataFrame, m: int = 4,
                         n_periods: int = 20) -> tuple:
    """
    Runs the full SARIMA pipeline for every column in df.

    Returns
    -------
    all_results     : dict {col: result_dict}  — use for dataset assembly
    all_comparisons : dict {col: comparison_df} — use for model diagnostics
    """
    all_results, all_comparisons = {}, {}

    for col in df.columns:
        print(f"\n{'='*60}\n  Processing: {col}\n{'='*60}")
        series        = df[col].dropna()
        comparison_df = fit_all_transformations(series, m=m)

        display_cols = ['transformation', 'aic', 'bic', 'order', 'seasonal_order',
                        'residual_std', 'prob_q', 'prob_jb', 'prob_h', 'skew', 'kurtosis']
        print(comparison_df[display_cols].to_string(index=False))

        result = forecast_best_model(comparison_df, n_periods=n_periods)
        print(f"\n  Best: [{result['transformation']}]  "
              f"SARIMA{result['order']}x{result['seasonal_order']}  "
              f"AIC={result['aic']:.2f}")

        all_results[col]     = result
        all_comparisons[col] = comparison_df

    return all_results, all_comparisons

## **2: Run SARIMA Forecasts**
___
Runs the SARIMA pipeline on all 12 raw series defined in `RAW_TO_FORECAST`.
For each series, three transformations (log, Box-Cox, Yeo-Johnson) are fitted
and compared by AIC. The best transformation is selected and used to generate
a 20-quarter forecast (2026 Q1 – 2030 Q4).

**Runtime:** approximately 5–20 minutes depending on machine.

**Output:** `results` dict — `{column: {'forecast': array, 'order': ..., ...}}`
This dict is consumed by Section 3. If you want to swap SARIMA for another
model, replace this cell with your model's fitting code and ensure `results`
has the same structure.

In [3]:
results, comparisons = run_sarima_pipeline(
    df_hist[RAW_TO_FORECAST],
    m=M,
    n_periods=N_PERIODS
)

print(f'\nCompleted SARIMA fitting for {len(results)} series.')
print('\nModel summary:')
print(f'  {"Series":<35s} {"Transform":<12s} {"Order":<15s} {"Seasonal":<20s} {"AIC":>8s}')
print(f'  {"-"*90}')
for col, res in results.items():
    print(f'  {col:<35s} {res["transformation"]:<12s} '
          f'{str(res["order"]):<15s} {str(res["seasonal_order"]):<20s} '
          f'{res["aic"]:>8.2f}')


  Processing: us_real_gdp
transformation         aic         bic     order seasonal_order  residual_std  prob_q  prob_jb  prob_h     skew  kurtosis
           log -879.314364 -870.425830 (0, 1, 1)   (0, 0, 0, 4)      0.767491  0.9259      0.0     0.0  11.8710  141.9492
   yeo-johnson -451.001013 -442.112479 (0, 1, 1)   (0, 0, 0, 4)      0.150986  0.9236      0.0     0.0 -10.1586  114.1747
        boxcox  883.049036  891.937570 (0, 1, 1)   (0, 0, 0, 4)     42.806644  0.9237      0.0     0.0  11.6028  137.8685

  Best: [log]  SARIMA(0, 1, 1)x(0, 0, 0, 4)  AIC=-879.31

  Processing: us_unemployment
transformation         aic         bic     order seasonal_order  residual_std  prob_q  prob_jb  prob_h    skew  kurtosis
        boxcox -713.013280 -707.073653 (1, 1, 0)   (0, 0, 0, 4)      0.076013  0.9876      0.0     0.0 10.7330  123.9318
           log -255.750833 -243.843898 (2, 0, 0)   (0, 0, 0, 4)      0.097106  0.9964      0.0     0.0  4.5639   48.7113
   yeo-johnson   82.012686   87.9

## **3: Derive Transformed Variables**
___
Reconstructs derived variables (YoY growth rates, log returns, first
differences) from the raw SARIMA forecasts. Derived variables require
historical context for the YoY/QoQ calculations — the historical raw series
is concatenated with the forecast period before computing each transformation.

**Why derive rather than forecast directly:**
Forecasting `us_gdp_yoy_growth` directly with SARIMA would treat it as an
independent series, ignoring its definitional relationship with `us_real_gdp`.
Deriving it from the GDP forecast preserves that relationship and ensures
internal consistency between the raw and derived forecasts.

**Output:** Derived variables are appended directly to `forecast_raw` in
memory and passed to Section 4 for assembly into the regressor dataset.

In [4]:
# ── Build forecast date index ─────────────────────────────────────────────────
last_date    = df_hist.index[-1]
forecast_idx = pd.date_range(
    start=last_date + pd.DateOffset(months=3),
    periods=N_PERIODS,
    freq='QE'
)

# ── Assemble raw forecasts ────────────────────────────────────────────────────
forecast_raw = pd.DataFrame(index=forecast_idx)
for col in RAW_TO_FORECAST:
    if col in results:
        forecast_raw[col] = results[col]['forecast']

# ── Reconstruct derived variables ─────────────────────────────────────────────
# Concatenate historical raw series with forecast period so that YoY/QoQ
combined = pd.concat([df_hist[RAW_TO_FORECAST], forecast_raw])

def build_derived(combined, source_col, method, periods, forecast_idx):
    """Compute a derived variable and slice out the forecast period."""
    if method == 'pct_change_yoy':
        return (combined[source_col].pct_change(periods) * 100).loc[forecast_idx]
    elif method == 'pct_change_qoq':
        return (combined[source_col].pct_change(periods) * 100).loc[forecast_idx]
    elif method == 'diff':
        return combined[source_col].diff(periods).loc[forecast_idx]
    elif method == 'log_diff':
        return np.log(combined[source_col]).diff(periods).loc[forecast_idx]
    else:
        raise ValueError(f'Unknown method: {method}')

for out_col, (src_col, method, periods) in DERIVED_VARS.items():
    forecast_raw[out_col] = build_derived(combined, src_col, method, periods, forecast_idx)

# ── Preview derived variables ────────────────────────────────────────────────
print(f'Derived variables constructed: {list(DERIVED_VARS.keys())}')
print(f'forecast_raw shape: {forecast_raw.shape}')
nan_counts = forecast_raw.isna().sum()
nan_cols   = nan_counts[nan_counts > 0]
if len(nan_cols) == 0:
    print('  No NaN values.')
else:
    print(f'  NaN values: {nan_cols.to_dict()}')

# ── Plot: all raw forecasts ───────────────────────────────────────────────────
n_cols = 3
n_rows = int(np.ceil(len(RAW_TO_FORECAST) / n_cols))

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[c.replace('us_', '').replace('_', ' ').title()
                    for c in RAW_TO_FORECAST],
    vertical_spacing=0.08, horizontal_spacing=0.06
)

for i, col in enumerate(RAW_TO_FORECAST):
    r, c = i // n_cols + 1, i % n_cols + 1
    hist_series = df_hist[col].dropna()
    fc_vals     = forecast_raw[col]

    fig.add_trace(go.Scatter(
        x=hist_series.index[-40:], y=hist_series.values[-40:],
        mode='lines', line=dict(color=NAVY, width=1.5),
        showlegend=(i == 0), name='Historical'), row=r, col=c)

    fig.add_trace(go.Scatter(
        x=fc_vals.index, y=fc_vals.values,
        mode='lines', line=dict(color=RED, width=1.5, dash='dash'),
        showlegend=(i == 0), name='SARIMA Forecast'), row=r, col=c)

fig.update_layout(
    title=dict(
        text='Figure 1 — SARIMA Forecasts: All Raw Series (2026–2030)<br>'
             '<sup>Last 10 years of history shown | Red dashed = forecast</sup>',
        font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=n_rows * 250,
    margin=dict(t=80, b=40, l=40, r=20))
fig.update_annotations(font_size=9)
fig.show()

Derived variables constructed: ['us_gdp_yoy_growth', 'us_house_price_yoy', 'us_indprod_yoy', 'us_oil_yoy', 'us_reer_diff', 'us_credit_qoq_growth', 'us_sp500_log_ret', 'us_vix_log_ret', 'us_bond_yield_10y_d1']
forecast_raw shape: (20, 21)
  NaN values: {'us_real_gdp': 1, 'us_credit': 2, 'us_house_price_idx': 1}


## **4: Assemble Complete Regressor Dataset**
___
Concatenates the full historical EDA dataset with the forecast-period raw
series, then recomputes all lagged columns on the combined series.

Lagged columns must be recomputed on the combined series — not pre-computed
separately for history and forecast — so that lag shifts correctly bridge the
boundary. For example, `us_house_price_yoy_L3` at 2026-Q1 is the value of
`us_house_price_yoy` at 2025-Q2, which is in the historical data.

**Lag structure from EDA Section 5 (Bank & Eder 2021, Section 10;**
**Djeundje & Crook 2019):**

| Variable | Lag | Transmission mechanism |
|---|---|---|
| `us_house_price_yoy` | L3 | Mortgage stress: 6–18 months |
| `us_consumer_confidence` | L2 | Leading indicator |
| `us_unemployment` | L1 | Lagging but within 1 year |
| `us_credit_qoq_growth` | L6 | Leverage build-up: 1–2 years |
| `us_gdp_yoy_growth` | L0 | Contemporaneous |
| `us_cpi` | L6 | Real income erosion |

**Output:** `SARIMA_regressors_US_Q.csv` — 164 rows × 42 columns saved to
`Data Collection/` locally and committed to the repository. This is the
single output of this notebook and the direct input for `Stage2_PD_Modelling.ipynb`.

In [5]:
# ── Concatenate historical + forecast ────────────────────────────────────────
# All columns retained — NaN in forecast rows for hist-only columns
combined_full = pd.concat([df_hist, forecast_raw], axis=0)

# ── Recompute lagged columns on combined series ───────────────────────────────
for var, lag in LAG_SPEC.items():
    if var in combined_full.columns:
        combined_full[f'{var}_L{lag}'] = combined_full[var].shift(lag)
    else:
        print(f'{var} not found in combined dataset — lag column skipped')

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = LOCAL_OUTPUT / OUT_REGRESSORS
combined_full.to_csv(out_path)
print(f'Saved {OUT_REGRESSORS}  {combined_full.shape}')
print(f'  Path            : {out_path}')
print(f'  Historical rows : {len(df_hist)}')
print(f'  Forecast rows   : {len(forecast_raw)}')
print(f'  Total rows      : {len(combined_full)}')
print(f'  Total columns   : {combined_full.shape[1]}')

# ── Validate: show forecast period lagged regressors ─────────────────────────
sig_lag_cols = [f"{v}_L{l}" for v, l in LAG_SPEC.items()
                if v in ['us_house_price_yoy', 'us_consumer_confidence',
                         'us_unemployment', 'us_credit_qoq_growth',
                         'us_gdp_yoy_growth', 'us_cpi']]
print('\nForecast period — Stage 2 significant regressors:')
print(combined_full.loc[forecast_idx, sig_lag_cols].round(4).to_string())

nan_check = combined_full.loc[forecast_idx, sig_lag_cols].isna().sum()
if nan_check.any():
    print('\nNaN values in significant regressors during forecast period:')
    print(nan_check[nan_check > 0])
else:
    print('\n✓ No NaN values in significant regressors during forecast period.')

Saved SARIMA_regressors_US_Q.csv  (165, 42)
  Path            : c:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Data Collection\SARIMA_regressors_US_Q.csv
  Historical rows : 145
  Forecast rows   : 20
  Total rows      : 165
  Total columns   : 42

Forecast period — Stage 2 significant regressors:
            us_house_price_yoy_L3  us_consumer_confidence_L2  us_unemployment_L1  us_credit_qoq_growth_L6  us_gdp_yoy_growth_L0  us_cpi_L6
2026-06-30                -0.3358                   100.4335              4.3000                  -0.2131                2.4984     2.8707
2026-09-30                -0.9569                   100.8895              4.2755                   0.4456                2.0281     2.3820
2026-12-31                -0.8975                   100.9468              4.2694                   0.6959                2.5298     2.6805
2027-03-31                -0.4416                   100.6994              4.2678                   0.9782                3.1579     

## **5: Forecast Summary & Validation**
___
Sanity checks and a summary plot of the six significant predictors from the
CCF analysis. These are the series that enter the Stage 2 PD regression at
their optimal lags — validating their forecast behaviour is a prerequisite
before running the modelling notebook.

**Key things to check:**
- Forecasts are economically plausible (no explosive paths)
- No flat lines at the last observed value (would indicate SARIMA(0,1,0)
  random walk — acceptable but noted)
- No NaN values in the significant regressor columns

In [6]:
# ── Significant predictors forecast plot ──────────────────────────────────────
SIG_VARS = {
    'us_house_price_yoy':     'House Price YoY (%)',
    'us_consumer_confidence': 'Consumer Confidence',
    'us_unemployment':        'Unemployment (%)',
    'us_credit_qoq_growth':   'Credit QoQ Growth (%)',
    'us_gdp_yoy_growth':      'GDP YoY Growth (%)',
    'us_cpi':                 'CPI',
}

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=list(SIG_VARS.values()),
    vertical_spacing=0.12, horizontal_spacing=0.07
)

for i, (var, label) in enumerate(SIG_VARS.items()):
    r, c = i // 3 + 1, i % 3 + 1

    hist_series = df_hist[var].dropna()
    fc_series   = forecast_raw[var] if var in forecast_raw.columns else None

    # Last 10 years of history
    cutoff = hist_series.index[-1] - pd.DateOffset(years=10)
    hist_plot = hist_series[hist_series.index >= cutoff]

    for s, e, lbl in RECESSIONS_US:
        if pd.Timestamp(s) >= cutoff:
            fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.3,
                          layer='below', line_width=0, row=r, col=c)

    fig.add_trace(go.Scatter(
        x=hist_plot.index, y=hist_plot.values,
        mode='lines', line=dict(color=NAVY, width=1.5),
        showlegend=(i == 0), name='Historical'), row=r, col=c)

    if fc_series is not None:
        fig.add_trace(go.Scatter(
            x=fc_series.index, y=fc_series.values,
            mode='lines', line=dict(color=RED, width=1.8, dash='dash'),
            showlegend=(i == 0), name='SARIMA Forecast'), row=r, col=c)

fig.update_layout(
    title=dict(
        text='Figure 2 — SARIMA Forecasts: CCF-Significant Predictors (2026–2030)<br>'
             '<sup>Last 10 years of history | Grey = recessions | '
             'These series enter Stage 2 at their CCF-optimal lags</sup>',
        font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1, font=dict(size=10)),
    margin=dict(t=100, b=40, l=50, r=20))
fig.update_annotations(font_size=10)
fig.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print('\nForecast Summary — Significant Predictors:')
print(f'  {"Variable":<35s} {"Hist last":>10s} {"FC mean":>10s} '
      f'{"FC min":>10s} {"FC max":>10s} {"Flat?":>6s}')
print(f'  {"-"*80}')
for var in SIG_VARS:
    if var not in forecast_raw.columns:
        continue
    hist_last = df_hist[var].dropna().iloc[-1]
    fc        = forecast_raw[var].dropna()
    flat      = 'Yes' if fc.std() < 0.001 else 'no'
    print(f'  {var:<35s} {hist_last:>10.3f} {fc.mean():>10.3f} '
          f'{fc.min():>10.3f} {fc.max():>10.3f} {flat:>6s}')

print(f'  Output : {OUT_REGRESSORS}')


Forecast Summary — Significant Predictors:
  Variable                             Hist last    FC mean     FC min     FC max  Flat?
  --------------------------------------------------------------------------------
  us_house_price_yoy                      -0.897     -0.062     -0.442     -0.010     no
  us_consumer_confidence                 100.889     99.930     99.711    100.947     no
  us_unemployment                          4.300      4.268      4.267      4.275     no
  us_credit_qoq_growth                     0.978      0.966      0.000      2.651     no
  us_gdp_yoy_growth                        1.989      2.458      1.849      3.158     no
  us_cpi                                   3.286      2.938      2.777      3.608     no
  Output : SARIMA_regressors_US_Q.csv
